# House Stage 1 regression: Model A vs Model A + a low-weight fundamentals blend

Plain Model A against a blend of A with a lean-inclusive fundamentals model, B, at a **capped, lower weight**, in the same expanding-window backtest (`01_reg_ensemble.ipynb` explored the alternatives):

- **Model A, polling:** `partisan_lean` + `gen_margin_raw` + `inc_signed` (fixed +2.5) + `partisan_lean:inc_signed` + `gb_momentum_30d_raw`.
- **Model B, fundamentals:** `partisan_lean` + `fundraising_advantage` + `pres_party_signed:is_midterm` + `pres_party_signed:cpi_region_log_yoy`. It uses no generic-ballot data (asserted). Both national terms enter **only signed by the president's party** (`a:b` is the product a × b), so each coefficient is the effect on the president's party's margin: the midterm penalty, and regional inflation (12-month log change). Gasoline is left out; its coefficient was about 0 once inflation was in.
- **The blend:** (1 − w) · A + w · B, with **w ≤ `WEIGHT_CAP`** (0.3, set before looking at the results).

**Choosing w without peeking:** the weight for eval year *t* is chosen on earlier windows only (2016 for 2018; 2016–2018 for 2020; 2016–2020 for 2022). It minimizes the mean of `WEIGHT_OBJECTIVE`, which weighs ex-ante MAE and winner accuracy **equally**, as relative changes against A alone:

`MAE(w) / MAE(0) + missed%(w) / missed%(0)` on `competitive_ex_ante`

So one extra missed race out of about 14 costs as much as a 7% higher MAE. The comparison varies w over the fixed weights `FIXED_WEIGHTS` (0.00, i.e. A alone, then 0.10, 0.15, 0.20), with the no-peeking weight alongside for reference. The fixed weights were picked after seeing earlier results, so they show how the blend responds to w; they are not a fair test of any one weight.

**Evaluation, in priority order:**

1. Ex-ante competitive (|lean + generic ballot| ≤ 10): MAE and winner accuracy, equally.
2. Ex-post competitive (|margin| ≤ 7.5): MAE and accuracy, as diagnostics.
3. All districts' MAE, and the residual correlation of A and B, as checks.
4. **Every competitive race where the blend changes A's predicted winner**, and whether that fixes a miss or breaks a correct call.

A and B fit the same rows in every window, each by **equal-year WLS** (`EQUAL_YEAR_WEIGHTS`): a training race is weighted by 1 / (the number of valid training races in its election year), so every training year counts the same in the fit. The weights are recomputed from each window's own training rows. **Caveat:** `fundraising_advantage`'s historical totals include post-election money, so B's contribution is somewhat optimistic. 2024 is never used.

In [1]:
%load_ext autoreload
%autoreload 2
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown

# The repository root is the first folder up from here that holds pyproject.toml, so this works at any depth
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))  # repository root, for experiments.house
import house.regression as hr  # hr.<Tab> lists everything; names are in src/house/regression/__init__.py
from experiments.house import ensemble, evaluation, tracking
from experiments.house.run import run_wls

In [2]:
import warnings

with warnings.catch_warnings(action="ignore"):  # loading warns with the same summaries printed below
    predictors = hr.load_predictors()  # data/processed/house/modeling.csv, every year (2006-2026)
    targets = hr.load_targets()        # data/processed/house/targets.csv, every year (2006-2024)
print(hr.validate_schema(predictors).summary())  # .issues lists every affected key

In [3]:
TARGET = "adj_margin_2p_norm"
ROLLING_WINDOWS = {  # eval year: its expanding training window
    2018: [2006, 2008, 2010, 2012, 2014, 2016],
    2020: [2006, 2008, 2010, 2012, 2014, 2016, 2018],
    2022: [2006, 2008, 2010, 2012, 2014, 2016, 2018, 2020],
}
# plus one earlier window, only to choose the blend weight for 2018 without peeking: it is never scored as a result
VALIDATION_WINDOWS = {2016: [2006, 2008, 2010, 2012, 2014]}
WINDOWS = {**VALIDATION_WINDOWS, **ROLLING_WINDOWS}
USE_HOLDOUT = False     # never True here: no window touches HOLDOUT_YEARS (2024)
# Evaluation only (training is unaffected): observed, non-imputed targets of these race types are scored ("valid")
EVAL_RACE_TYPES = ["dr", "dr_multi", "dr_jungle", "dr_rcv", "dr_writein"]
# Competitive subsets of the valid targets: name -> (column, threshold); a row is in one when abs(column) <= threshold.
# column is a column name or arithmetic of them, from the targets or the predictors
EX_ANTE_THRESHOLD = 10   # points: any non-negative int or float, e.g. 5, 7, or 7.5
EX_POST_THRESHOLD = 7.5
COMPETITIVE = {
    # expected margin under uniform swing: two-party lean (2 x cook_pvi) + eve-of-election two-party generic ballot
    "competitive_ex_ante": ("partisan_lean + gb_margin_2p_norm", EX_ANTE_THRESHOLD),
    "competitive_ex_post": ("margin_2p_norm", EX_POST_THRESHOLD),  # observed two-party margin
}
SUMMARY_SECTIONS = {"ALL": "valid", "EX_ANTE_COMPETITIVE": "competitive_ex_ante",
                    "EX_POST_COMPETITIVE": "competitive_ex_post"}
RANK_BY = "mae"     # variants ranked by mean out-of-sample MAE ...
TIEBREAK = "rmse"   # ... ties broken by RMSE

## Models

In [4]:
FIXED_TERM = "inc_signed"
MODEL_A_FIXED_INC = 2.5
# Both models weight each training race by 1 / (the valid training races of its election year), so every training
# year carries the same total weight (run_wls's equal_year; recomputed from each window's own training rows)
EQUAL_YEAR_WEIGHTS = True

# Known before every election: the White House party (+1 D, -1 R) and whether the election is a midterm
PRES_PARTY_SIGNED = {2006: -1, 2008: -1, 2010: 1, 2012: 1, 2014: 1, 2016: 1, 2018: -1, 2020: -1, 2022: 1, 2024: 1,
                     2026: -1}
predictors = predictors.assign(pres_party_signed=predictors["year"].map(PRES_PARTY_SIGNED),
                               is_midterm=(predictors["year"] % 4 == 2).astype(int))

MODELS = {
    "A_polling": (f"{TARGET} ~ partisan_lean + gen_margin_raw + inc_signed + partisan_lean:inc_signed"
                  " + gb_momentum_30d_raw", {FIXED_TERM: MODEL_A_FIXED_INC}),
    "B_fundamentals": (f"{TARGET} ~ partisan_lean + fundraising_advantage + pres_party_signed:is_midterm"
                       " + pres_party_signed:cpi_region_log_yoy", {}),  # OLS
}
A, B = MODELS  # the model names

GB_COLUMNS = ["gen_margin_raw", *(column for column in predictors.columns if column.startswith("gb_"))]
_, b_variables = evaluation.formula_variables(MODELS[B][0], predictors.columns)
assert not set(b_variables) & set(GB_COLUMNS), "generic-ballot column in Model B"
# every national economic / election-type term in B enters only multiplied by pres_party_signed
from patsy import ModelDesc

b_terms = [term.name() for term in ModelDesc.from_formula(MODELS[B][0]).rhs_termlist]
assert all(term.startswith("pres_party_signed:") for term in b_terms
           if any(v in term for v in ["is_midterm", "cpi_region_log_yoy"])), b_terms
assert "pres_party_signed" not in b_terms and "cpi_region_log_yoy" not in b_terms, b_terms
SHARED_COLUMNS = evaluation.shared_columns(formulas=[formula for formula, _ in MODELS.values()],
                                           columns=predictors.columns)
print("shared columns:", SHARED_COLUMNS)
for name, (formula, fixed) in MODELS.items():
    print(f"{name:16s} {formula}   fixed: {fixed or 'none (OLS)'}")

shared columns: ['partisan_lean', 'gen_margin_raw', 'inc_signed', 'gb_momentum_30d_raw', 'fundraising_advantage', 'pres_party_signed', 'is_midterm', 'cpi_region_log_yoy']
A_polling        adj_margin_2p_norm ~ partisan_lean + gen_margin_raw + inc_signed + partisan_lean:inc_signed + gb_momentum_30d_raw   fixed: {'inc_signed': 2.5}
B_fundamentals   adj_margin_2p_norm ~ partisan_lean + fundraising_advantage + pres_party_signed:is_midterm + pres_party_signed:cpi_region_log_yoy   fixed: none (OLS)


In [5]:
# Every window (2016 only chooses 2018's weight): A and B on the same rows. The missing-predictor warnings (FEC has
# no data for a few winners) repeat; each run's exclusions are in its .excluded
with warnings.catch_warnings(action="ignore"):
    fits = {
        eval_year: {
            name: run_wls(
                formula=formula,
                train_years=train_years,
                eval_years=[eval_year],
                predict_years=[],
                equal_year=EQUAL_YEAR_WEIGHTS,
                fixed=fixed,
                predictors=predictors,
                targets=targets,
                use_holdout=USE_HOLDOUT,
                require_complete=SHARED_COLUMNS,
                race_types=EVAL_RACE_TYPES,
                competitive=COMPETITIVE,
                name=name,
                save=False,
            )
            for name, (formula, fixed) in MODELS.items()
        }
        for eval_year, train_years in WINDOWS.items()
    }
for eval_year, runs in fits.items():  # matched samples
    assert len({(r.counts["train_rows_hash"], r.counts["eval_rows_hash"]) for r in runs.values()}) == 1, eval_year

## Residual correlation (check)

The correlation of A's and B's residuals (actual − predicted) on the same districts, by eval year and pooled, with each model's bias and MAE. Lower correlation leaves more for a blend to gain.

In [6]:
SUBSETS = list(SUMMARY_SECTIONS.values())
pairs = {year: [(fits[year][A], fits[year][B])] for year in ROLLING_WINDOWS}
pairs["pooled"] = [(fits[year][A], fits[year][B]) for year in ROLLING_WINDOWS]
correlations = pd.DataFrame({key: ensemble.error_correlation(results, SUBSETS).stack() for key, results in pairs.items()}).T
correlations.index.name = "eval_year"
correlations.style.format({column: "{:.0f}" if column[1] == "n" else "{:+.2f}" if column[1].startswith("bias")
                           else "{:.2f}" for column in correlations.columns})

## Choosing the weight

The blend is scored at every weight on the grid, in every window. `WEIGHT_OBJECTIVE` is computed per window against that window's A alone (w = 0). The rule then picks, for each eval year, the capped weight with the lowest mean objective over the earlier windows.

In [7]:
WEIGHT_GRID = np.round(np.arange(0, 1.0001, 0.05), 2)  # w on Model B
WEIGHT_CAP = 0.3                    # "lower weighting": B never gets more than this
WEIGHT_SUBSET = "competitive_ex_ante"
FIXED_WEIGHTS = [0.00, 0.10, 0.15, 0.20]  # the weights compared side by side (0.00 is A alone)


def with_objective(table: pd.DataFrame) -> pd.DataFrame:
    """A window's blend scores plus `objective` on WEIGHT_SUBSET: MAE and missed % each relative to A alone (w = 0),
    summed, so a 7% higher MAE costs as much as one more missed race in 14."""
    subset = table.xs(WEIGHT_SUBSET, level="subset")
    objective = subset["mae"] / subset.loc[0.0, "mae"] + subset["missed_pct"] / subset.loc[0.0, "missed_pct"]
    return table.join(pd.Series(objective.values, index=pd.MultiIndex.from_product(
        [objective.index, [WEIGHT_SUBSET]], names=["weight", "subset"]), name="objective"))


with warnings.catch_warnings(action="ignore"):
    scores = {year: with_objective(ensemble.blend_scores(fits[year][A], fits[year][B], WEIGHT_GRID, targets,
                                                         predictors, TARGET, COMPETITIVE, EVAL_RACE_TYPES))
              for year in WINDOWS}
capped = {year: table[table.index.get_level_values("weight") <= WEIGHT_CAP] for year, table in scores.items()}

CHOSEN = {}  # eval year: (weight, the windows it was chosen on)
for year in ROLLING_WINDOWS:
    CHOSEN[year] = ensemble.choose_weight(capped, year, WEIGHT_SUBSET, "objective")
pd.DataFrame({year: {"weight": w, "chosen on": ", ".join(map(str, earlier))} for year, (w, earlier) in CHOSEN.items()}).T

,weight,chosen on
2018,0.3,2016
2020,0.3,"2016, 2018"
2022,0.3,"2016, 2018, 2020"


## Plain A vs the blend

For each eval year, the blend at each fixed weight (w = 0.00 is A alone) and at the no-peeking weight, then the mean of the three years, each weighted equally. Sections are in priority order; Δ is blend − A on the same districts. ✓ marks an improvement and ✗ a regression.

In [8]:
SECTIONS = {"EX_ANTE": "competitive_ex_ante", "EX_POST": "competitive_ex_post", "ALL": "valid"}  # priority order
COMPARE_METRICS = ["n", "mae", "rmse", "bias", "accuracy_pct", "missed_races"]
DELTA_METRICS = ["mae", "rmse", "accuracy_pct", "missed_races"]


def compare_row(year, weight):
    row = {}
    for section, subset in SECTIONS.items():
        blended, alone = scores[year].loc[(weight, subset)], scores[year].loc[(0.0, subset)]
        row |= {(section, m): blended[m] for m in COMPARE_METRICS}
        row |= {(section, f"Δ{m}"): blended[m] - alone[m] for m in DELTA_METRICS}
    return row


BLENDS = {  # label: the weight on B in each eval year
    **{("w = 0.00 (A alone)" if w == 0 else f"w = {w:.2f}"): {year: w for year in ROLLING_WINDOWS} for w in FIXED_WEIGHTS},
    f"no-peek (w ≤ {WEIGHT_CAP:g})": {year: CHOSEN[year][0] for year in ROLLING_WINDOWS},
}
rows = {}
for label, weights in BLENDS.items():
    for year in ROLLING_WINDOWS:
        rows[(label, year)] = compare_row(year, weights[year]) | {("weight", "w"): weights[year]}
comparison = pd.DataFrame.from_dict(rows, orient="index").rename_axis(["model", "eval_year"])
comparison = comparison[[("weight", "w"), *(c for c in comparison.columns if c[0] != "weight")]]
mean = comparison.groupby(level="model", sort=False).mean()
mean.index = pd.MultiIndex.from_tuples([(m, "mean") for m in mean.index], names=comparison.index.names)
comparison = pd.concat([comparison, mean])
order = list(dict.fromkeys(comparison.index.get_level_values("model")))
comparison = comparison.reindex([(m, y) for m in order for y in [*ROLLING_WINDOWS, "mean"]])
tracking.style_best_of(comparison)

## Winner flips in competitive races

Every ex-ante or ex-post competitive race where the no-peeking blend calls a different winner than A, with each model's predicted margin. `verdict`:

- **fixed:** the blend is right and A was wrong.
- **broken:** A was right and the blend is wrong.

Margins are D − R, in points. These flips are where the blend's accuracy change comes from.

In [9]:
flips = []
for label, weights in BLENDS.items():
    if label.startswith("w = 0.00"):
        continue  # A itself: no flips
    for year in ROLLING_WINDOWS:
        weight = weights[year]
        _, _, blended = evaluation.evaluate_predictions(ensemble.blend(fits[year][A], fits[year][B], weight), targets,
                                                        predictors, TARGET, competitive=COMPETITIVE,
                                                        race_types=EVAL_RACE_TYPES)
        a = fits[year][A].scored
        b = fits[year][B].scored[[*evaluation.KEY, "predicted_margin"]].rename(columns={"predicted_margin": "B"})
        merged = a.merge(blended[[*evaluation.KEY, "predicted_margin", "predicted_winner"]], on=evaluation.KEY,
                         suffixes=("", "_blend")).merge(b, on=evaluation.KEY)
        changed = merged[(merged["predicted_winner"] != merged["predicted_winner_blend"])
                         & (merged["competitive_ex_ante"] | merged["competitive_ex_post"])]
        for r in changed.itertuples():
            flips.append({
                "blend": label, "eval_year": year, "district": evaluation.district_label(r.state, r.district),
                "w": weight, "ex_ante": r.competitive_ex_ante, "ex_post": r.competitive_ex_post,
                "actual": r.actual_margin, "A": r.predicted_margin, "B": r.B, "blended": r.predicted_margin_blend,
                "winner": r.actual_winner, "A_called": r.predicted_winner, "blend_called": r.predicted_winner_blend,
                "verdict": "fixed" if r.predicted_winner_blend == r.actual_winner else "broken",
            })
flips = pd.DataFrame(flips, columns=["blend", "eval_year", "district", "w", "ex_ante", "ex_post", "actual", "A", "B",
                                     "blended", "winner", "A_called", "blend_called", "verdict"])
summary = (flips.groupby(["blend", "verdict"], sort=False).size().unstack(fill_value=0)
           .reindex(index=[label for label in BLENDS if not label.startswith("w = 0.00")], columns=["fixed", "broken"],
                    fill_value=0))
summary["net"] = summary["fixed"] - summary["broken"]
display(Markdown("### Winner flips in competitive races, by blend (fixed − broken = net change in correct calls)"))
display(summary)
flips.style.format({"w": "{:.2f}", "actual": "{:+.1f}", "A": "{:+.1f}", "B": "{:+.1f}", "blended": "{:+.1f}"}).map(
    lambda v: f"color: {tracking.GOOD}" if v == "fixed" else f"color: {tracking.BAD}" if v == "broken" else "",
    subset=["verdict"])

### Winner flips in competitive races, by blend (fixed − broken = net change in correct calls)

verdict,fixed,broken,net
blend,,,
w = 0.10,2,1,1
w = 0.15,4,2,2
w = 0.20,4,3,1
no-peek (w ≤ 0.3),6,6,0


,blend,eval_year,district,w,ex_ante,ex_post,actual,A,B,blended,winner,A_called,blend_called,verdict
0,w = 0.10,2018,VA-02,0.10,True,True,+2.2,-0.4,+4.2,+0.1,D,R,D,fixed
1,w = 0.10,2022,CO-08,0.10,True,True,+0.7,+0.0,-2.0,-0.2,D,D,R,broken
2,w = 0.10,2022,NY-19,0.10,True,True,-1.6,+0.1,-2.8,-0.2,R,D,R,fixed
3,w = 0.15,2018,NE-02,0.15,True,True,-2.0,-0.7,+3.9,+0.0,R,R,D,broken
4,w = 0.15,2018,NJ-03,0.15,True,True,+1.3,-0.8,+5.5,+0.2,D,R,D,fixed
5,w = 0.15,2018,NY-19,0.15,True,True,+5.3,-1.0,+8.4,+0.4,D,R,D,fixed
6,w = 0.15,2018,VA-02,0.15,True,True,+2.2,-0.4,+4.2,+0.3,D,R,D,fixed
7,w = 0.15,2022,CO-08,0.15,True,True,+0.7,+0.0,-2.0,-0.3,D,D,R,broken
8,w = 0.15,2022,NY-19,0.15,True,True,-1.6,+0.1,-2.8,-0.3,R,D,R,fixed
9,w = 0.20,2018,NE-02,0.20,True,True,-2.0,-0.7,+3.9,+0.2,R,R,D,broken


## Coefficients

Each model's coefficients averaged over the three eval windows; in Model A, `inc_signed` is the imposed 2.5.

In [10]:
pd.DataFrame({name: sum(fits[year][name].coefficients["coef"] for year in ROLLING_WINDOWS) / len(ROLLING_WINDOWS)
              for name in MODELS}).round(3)

,A_polling,B_fundamentals
term,,
Intercept,-0.475,2.240
fundraising_advantage,NaN,5.128
gb_momentum_30d_raw,-1.494,NaN
gen_margin_raw,1.268,NaN
inc_signed,2.500,NaN
partisan_lean,1.066,0.661
partisan_lean:inc_signed,-0.102,NaN
pres_party_signed:cpi_region_log_yoy,NaN,-0.545
pres_party_signed:is_midterm,NaN,-3.221
